In [1]:
# Cài đặt Unsloth và các thư viện cần thiết
!pip install -q unsloth "trl>=0.9" "transformers>=4.44" "datasets>=2.20"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 23.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 106.7 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 44.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 61.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 73.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 86.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 70.3 MB/s eta 0:00:00:00:01
  

In [2]:
# ===== HYPERPARAMETERS =====
MODEL_NAME      = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH  = 2048
LOAD_IN_4BIT    = True

# Cấu hình LoRA
LORA_R          = 16
LORA_ALPHA      = 16
LORA_DROPOUT    = 0.0

# Cấu hình Training
LEARNING_RATE   = 2e-4
EPOCHS          = 3       # Đã cập nhật thành 3 Epochs
BATCH_SIZE      = 4
GRAD_ACCUM      = 2
WARMUP_STEPS    = 5
SEED            = 42

# Cấu hình Dữ liệu
DATASET_NAME    = "klusai/ds-tf1-en-3m"
NUM_RECORDS     = 1000    # Chỉ lấy 1000 mẫu

# Đầu ra
OUTPUT_DIR      = "/kaggle/working/outputs"
GGUF_OUTPUT_DIR = "/kaggle/working/llama3-fable-1000-gguf"

In [3]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.5: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-3B-Instruct-bnb-4bit as a legacy tokenizer.
Unsloth 2026.7.5 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [4]:
from datasets import load_dataset

print("Đang tải và xử lý dữ liệu...")
raw_ds = load_dataset(DATASET_NAME, split=f"train[:{NUM_RECORDS}]")
split_ds = raw_ds.train_test_split(test_size=0.1, seed=SEED)

train_data = split_ds["train"]
val_data = split_ds["test"]

def to_text(ex):
    messages = [
        {"role": "system", "content": ex["system_message"]},
        {"role": "user", "content": ex["prompt"]},
        {"role": "assistant", "content": ex["fable"]},
    ]
    # Hàm này gộp 3 trường lại thành chuẩn định dạng của Llama 3
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": text}

train_ds = train_data.map(to_text, num_proc=2)
val_ds = val_data.map(to_text, num_proc=2)

# Lưu JSON
train_ds.to_json("/kaggle/working/train_data_report.json", force_ascii=False)
val_ds.to_json("/kaggle/working/val_data_report.json", force_ascii=False)

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}.")

Đang tải và xử lý dữ liệu...


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

klusai-ds-tf1-en-3m_train_00000.parquet:   0%|          | 0.00/100M [00:00<?, ?B/s]

...

Map (num_proc=2):   0%|          | 0/900 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/100 [00:00<?, ? examples/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Kích thước tập Train: 900 mẫu, Val: 100 mẫu.


In [5]:
import torch
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    args=SFTConfig(
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=WARMUP_STEPS,
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        logging_steps=5,             # Cập nhật log mỗi 10 steps
        eval_strategy="epoch",        # Chạy validation sau mỗi epoch
        output_dir=OUTPUT_DIR,
        seed=SEED,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        report_to="none",
    ),
)

print("Bắt đầu quá trình huấn luyện toàn diện...")
trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/900 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/100 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Bắt đầu quá trình huấn luyện toàn diện...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 900 | Num Epochs = 3 | Total steps = 339
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Epoch,Training Loss,Validation Loss
1,0.492277,0.488285
2,0.427686,0.452822
3,0.396075,0.451421


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/outputs/checkpoint-339/tokenizer_config.json.


TrainOutput(global_step=339, training_loss=0.5147292473323226, metrics={'train_runtime': 2837.4388, 'train_samples_per_second': 0.952, 'train_steps_per_second': 0.119, 'total_flos': 3.1294905709400064e+16, 'train_loss': 0.5147292473323226, 'epoch': 3.0})

In [6]:
# Export thẳng ra GGUF lượng tử hóa 4-bit, chuẩn tốt nhất cho 3B
model.save_pretrained_gguf(
    GGUF_OUTPUT_DIR, 
    tokenizer, 
    quantization_method="q4_k_m" 
)

print(f"Hoàn thành! Bạn có thể tải file GGUF tại tab Data (bên phải Kaggle) -> Output -> {GGUF_OUTPUT_DIR}")

Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/llama3-fable-1000-gguf/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


...

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [00:17<00:00,  8.61s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [00:48<00:00, 24.03s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/llama3-fable-1000-gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b10079-mix-fb3d4ca (app-b10079-mix-fb3d4ca-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/kaggle/working/llama3-fable-1000-gguf_gguf/Llama-3.2-3B-Instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
U

In [7]:
!zip -r /kaggle/working/outputs.zip /kaggle/working/outputs

  adding: kaggle/working/outputs/ (stored 0%)
  adding: kaggle/working/outputs/checkpoint-339/ (stored 0%)
  adding: kaggle/working/outputs/checkpoint-339/trainer_state.json (deflated 77%)
  adding: kaggle/working/outputs/checkpoint-339/tokenizer_config.json (deflated 96%)
  adding: kaggle/working/outputs/checkpoint-339/rng_state.pth (deflated 26%)
  adding: kaggle/working/outputs/checkpoint-339/chat_template.jinja (deflated 71%)
  adding: kaggle/working/outputs/checkpoint-339/README.md (deflated 65%)
  adding: kaggle/working/outputs/checkpoint-339/scheduler.pt (deflated 62%)
  adding: kaggle/working/outputs/checkpoint-339/optimizer.pt (deflated 11%)
  adding: kaggle/working/outputs/checkpoint-339/training_args.bin (deflated 53%)
  adding: kaggle/working/outputs/checkpoint-339/scaler.pt (deflated 64%)
  adding: kaggle/working/outputs/checkpoint-339/adapter_model.safetensors (deflated 8%)
  adding: kaggle/working/outputs/checkpoint-339/tokenizer.json (deflated 85%)
  adding: kaggle/work

In [9]:
!zip -r /kaggle/working/llama3-fable-1000-gguf.zip /kaggle/working/llama3-fable-1000-gguf

  adding: kaggle/working/llama3-fable-1000-gguf/ (stored 0%)
  adding: kaggle/working/llama3-fable-1000-gguf/model-00002-of-00002.safetensors (deflated 21%)
  adding: kaggle/working/llama3-fable-1000-gguf/model-00001-of-00002.safetensors (deflated 21%)
  adding: kaggle/working/llama3-fable-1000-gguf/model.safetensors.index.json (deflated 96%)
  adding: kaggle/working/llama3-fable-1000-gguf/config.json (deflated 57%)
  adding: kaggle/working/llama3-fable-1000-gguf/generation_config.json (deflated 37%)
  adding: kaggle/working/llama3-fable-1000-gguf/tokenizer_config.json (deflated 94%)
  adding: kaggle/working/llama3-fable-1000-gguf/chat_template.jinja (deflated 71%)
  adding: kaggle/working/llama3-fable-1000-gguf/.cache/ (stored 0%)
  adding: kaggle/working/llama3-fable-1000-gguf/.cache/huggingface/ (stored 0%)
  adding: kaggle/working/llama3-fable-1000-gguf/.cache/huggingface/.gitignore (stored 0%)
  adding: kaggle/working/llama3-fable-1000-gguf/.cache/huggingface/CACHEDIR.TAG (deflate